# CodePilot Real Dataset Fine-Tuning
Train Qwen2.5-Coder-1.5B on real-world code review data using Unsloth.

In [ ]:
!pip install -q unsloth trl transformers datasets torch
!pip install --no-deps xformers "unsloth_zoo[conda] @ git+https://github.com/unslothai/unsloth-zoo.git"

In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

SYSTEM_PROMPT = "You are an expert Python code reviewer. Review the provided code snippet for security vulnerabilities, bugs, performance issues, error handling flaws, and resource leaks. Return ONLY valid JSON matching the required schema with line numbers, category, severity, issue description, and fix. Return only raw JSON string, NO markdown code blocks."

def formatting_prompts_func(example):
    messages = example["messages"]
    messages[0]["content"] = SYSTEM_PROMPT
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = load_dataset("json", data_files={"train": "train_real.jsonl", "validation": "val_real.jsonl"})
dataset = dataset.map(formatting_prompts_func)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset = dataset["validation"],
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 300,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        output_dir = "outputs_real",
    ),
)

trainer_stats = trainer.train()

In [ ]:
model.save_pretrained_gguf("codepilot-real-1.5b", tokenizer, quantization_method = "q4_k_m")